# NAS Lee Kuan Yew speeches scraper (Colab)

Collects speech PDFs from `https://www.nas.gov.sg/archivesonline/data/pdfdoc/lky########.pdf`, extracts the text, and writes a JSONL file for your RAG pipeline.

**Personal / non-commercial use only.** NAS permits viewing but requires written permission for reproduction. Keep the output private and keep the source URL as metadata.

Two ways to find the PDFs (use A, B, or both):
- **A. Harvest** document numbers with a regex from NAS pages you give it (fast, few requests).
- **B. Probe** every calendar date in a year range (slow, fallback when A misses speeches).

Run the cells top to bottom.

In [ ]:
!pip -q install requests pypdf tqdm

In [ ]:
import os

# Persist results on Google Drive so a Colab disconnect doesn't lose progress
try:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKDIR = '/content/drive/MyDrive/lky_nas'
except Exception:
    WORKDIR = '/content/lky_nas'
os.makedirs(WORKDIR + '/pdfs', exist_ok=True)
os.makedirs(WORKDIR + '/saved_pages', exist_ok=True)   # optional: saved NAS HTML pages for mode A

# ---- settings ----
YEAR_START, YEAR_END = 1959, 2011      # range for mode B (narrow it for faster runs)
SUFFIXES = ['']                        # e.g. ['', 'a'] if some doc numbers carry a trailing letter
PROBE_SUFFIX_ON_HIT = list('abcde')    # after a date hits, also test lky<date>a..e
WORKERS = 3                            # keep low, this is a government archive
DELAY = 1.0                            # seconds each worker waits before every request
CONTACT = 'your-email@example.com'     # put your real email so NAS can contact you

In [ ]:
import re, time, random, json, threading, datetime as dt
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed
from urllib.robotparser import RobotFileParser
from tqdm.auto import tqdm

BASE = 'https://www.nas.gov.sg/archivesonline/data/pdfdoc/'
UA = f'Mozilla/5.0 (personal research; non-commercial; contact: {CONTACT})'

# lky + 8-digit date (YYYYMMDD) + optional 1-char suffix.  Matches both PDF hrefs and the
# "Document Number: lky19921214" text shown on record pages.
DOCNUM_RE = re.compile(r'\blky\d{8}[a-z0-9]?\b', re.I)
PDF_RE = re.compile(r'/archivesonline/data/pdfdoc/(lky\d{8}[a-z0-9]?)\.pdf', re.I)

# robots.txt check
rp = RobotFileParser()
try:
    rp.set_url('https://www.nas.gov.sg/robots.txt'); rp.read()
except Exception:
    rp = None
def allowed(url): return True if rp is None else rp.can_fetch(UA, url)
print('robots.txt allows pdfdoc path:', allowed(BASE + 'lky19921214.pdf'))

_local = threading.local()
def session():
    if not hasattr(_local, 's'):
        _local.s = requests.Session(); _local.s.headers['User-Agent'] = UA
    return _local.s

def is_pdf(url):
    # True = real PDF, False = not found / not a PDF, None = network trouble (retry next run)
    for attempt in range(3):
        try:
            time.sleep(DELAY + random.random() * 0.5)
            r = session().get(url, stream=True, timeout=30)
            code = r.status_code
            if code == 200:
                head = next(r.iter_content(8), b'')
                r.close()
                return head.startswith(b'%PDF')      # catches soft-404 HTML pages
            r.close()
            if code in (429, 503):
                time.sleep(30 * (attempt + 1)); continue
            return False
        except requests.RequestException:
            time.sleep(5 * (attempt + 1))
    return None

# resumable state
checked_f, hits_f = WORKDIR + '/checked.txt', WORKDIR + '/hits.txt'
def load(p): return set(open(p).read().split()) if os.path.exists(p) else set()
checked, hits = load(checked_f), load(hits_f)
print(f'state: {len(checked)} ids checked, {len(hits)} hits so far')

In [ ]:
# Sanity check: paste ONE real PDF link copied from a NAS record page ("Click to view text of speech")
SAMPLE_URL = ''   # e.g. 'https://www.nas.gov.sg/archivesonline/data/pdfdoc/lky19921214.pdf'
if SAMPLE_URL:
    print('is a real PDF:', is_pdf(SAMPLE_URL))
    m = PDF_RE.search(SAMPLE_URL)
    print('doc id parsed:', m.group(1) if m else 'NO MATCH, check the URL pattern / SUFFIXES')

## A. Harvest document numbers with the regex
Give it NAS search-result or record pages (URLs), and/or save pages from your browser (Ctrl+S, "Webpage, HTML only") into `saved_pages/` in the work folder. It extracts every `lky########` id it finds.

In [ ]:
import glob

PAGE_URLS = [
    # 'https://www.nas.gov.sg/archivesonline/speeches/...search results URL...',
]

found = set()
for u in PAGE_URLS:
    if not allowed(u):
        print('robots.txt disallows', u); continue
    time.sleep(DELAY)
    html = session().get(u, timeout=30).text
    found |= {m.lower() for m in DOCNUM_RE.findall(html)}
for p in glob.glob(WORKDIR + '/saved_pages/*.htm*'):
    found |= {m.lower() for m in DOCNUM_RE.findall(open(p, errors='ignore').read())}

new = found - hits
with open(hits_f, 'a') as hf:
    for i in sorted(new): hf.write(i + '\n')
hits |= new
print(f'harvested {len(found)} ids ({len(new)} new). total hits: {len(hits)}')

## B. Probe every date (fallback)
Requests one URL per calendar day in the year range (about 365 per year, so 1959-2011 is roughly 19,000 requests, about 2 hours at the default pace). Resumable: if Colab disconnects, re-run and it continues. Start with a narrow range (e.g. 1965 to 1965) to confirm it works.

In [ ]:
def all_dates(y0, y1):
    d, end = dt.date(y0, 1, 1), dt.date(y1, 12, 31)
    while d <= end:
        yield d.strftime('%Y%m%d'); d += dt.timedelta(days=1)

def probe(doc_id):
    return doc_id, is_pdf(f'{BASE}{doc_id}.pdf')

def run_probe(ids):
    todo = [i for i in ids if i not in checked]
    if not todo:
        print('nothing to do'); return
    with ThreadPoolExecutor(WORKERS) as ex, open(checked_f, 'a') as cf, open(hits_f, 'a') as hf:
        futs = [ex.submit(probe, i) for i in todo]
        for f in tqdm(as_completed(futs), total=len(futs)):
            i, ok = f.result()
            if ok is None: continue
            cf.write(i + '\n'); checked.add(i); cf.flush()
            if ok and i not in hits:
                hf.write(i + '\n'); hits.add(i); hf.flush()

if allowed(BASE + 'lky19921214.pdf'):
    ids = [f'lky{d}{s}' for d in all_dates(YEAR_START, YEAR_END) for s in SUFFIXES]
    run_probe(ids)
    # second pass: same-day extras like lky19921214a..e
    base_hits = [h for h in sorted(hits) if len(h) == 11]
    run_probe([h + s for h in base_hits for s in PROBE_SUFFIX_ON_HIT])
    print(f'total hits: {len(hits)}')
else:
    print('robots.txt disallows this path. Stop here, or use the record-page script instead.')

## Download PDFs and extract text
Writes `lky_speeches.jsonl` (one speech per line). PDFs with almost no extractable text are likely scanned images and are listed in `needs_ocr.txt`.

In [ ]:
from pypdf import PdfReader

def clean(t):
    t = re.sub(r'-\n(?=[a-z])', '', t)      # de-hyphenate line breaks
    t = re.sub(r'[ \t]+', ' ', t)
    t = re.sub(r'\n{3,}', '\n\n', t)
    return t.strip()

out_f = WORKDIR + '/lky_speeches.jsonl'
done = set()
if os.path.exists(out_f):
    for line in open(out_f):
        try: done.add(json.loads(line)['doc_id'])
        except Exception: pass

ocr_needed = []
with open(out_f, 'a', encoding='utf-8') as fh:
    for doc_id in tqdm(sorted(hits)):
        if doc_id in done: continue
        url = f'{BASE}{doc_id}.pdf'
        path = f'{WORKDIR}/pdfs/{doc_id}.pdf'
        if not os.path.exists(path):
            time.sleep(DELAY)
            r = session().get(url, timeout=60)
            if r.status_code != 200 or not r.content.startswith(b'%PDF'):
                print('skip (not a PDF):', url); continue
            open(path, 'wb').write(r.content)
        try:
            reader = PdfReader(path)
            text = clean('\n'.join((p.extract_text() or '') for p in reader.pages))
            pages = len(reader.pages)
        except Exception as e:
            print('PDF error', doc_id, e); continue
        if len(text) < 200:
            ocr_needed.append(doc_id)
        try:
            date = dt.datetime.strptime(doc_id[3:11], '%Y%m%d').date().isoformat()
        except ValueError:
            date = None
        first_line = next((l.strip() for l in text.split('\n') if l.strip()), '')
        rec = {'doc_id': doc_id, 'date': date, 'url': url, 'pages': pages,
               'title_guess': first_line[:200], 'speaker': 'Lee Kuan Yew',
               'source': 'National Archives of Singapore, Archives Online', 'text': text}
        fh.write(json.dumps(rec, ensure_ascii=False) + '\n'); fh.flush()

open(WORKDIR + '/needs_ocr.txt', 'a').write('\n'.join(ocr_needed) + ('\n' if ocr_needed else ''))
print(f'done. {len(ocr_needed)} PDFs look scanned (need OCR). Output: {out_f}')

In [ ]:
# Preview + download
rows = [json.loads(l) for l in open(out_f, encoding='utf-8')]
print(len(rows), 'speeches;', sum(len(r['text']) for r in rows), 'characters total')
for r in rows[:3]:
    print(r['doc_id'], r['date'], r['title_guess'][:80], '|', len(r['text']), 'chars')
try:
    from google.colab import files
    files.download(out_f)
except Exception:
    pass

**If many PDFs need OCR:** install `ocrmypdf` (`!apt -qq install ocrmypdf`) and run `ocrmypdf --force-ocr in.pdf out.pdf` on the files listed in `needs_ocr.txt`, then delete those ids from `lky_speeches.jsonl` and re-run the extraction cell.